In [1]:
!pip install lazypredict


   -------------------- ------------------- 1/2 [lazypredict]
   -------------------- ------------------- 1/2 [lazypredict]
   ---------------------------------------- 2/2 [lazypredict]



In [ ]:
# ===============================
# 1. Import
# ===============================
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyRegressor

# ===============================
# 2. Load Data
# ===============================
engine = create_engine("sqlite:///D:/FPT_SP26/ADY201m/Project_Clone/sample_v4.db")

     
df = pd.read_sql(
    'select * from NearsestSample where TimeDim >= 2000 AND SpatialDim = "VNM" order by TimeDim',
    engine
)

df.drop(columns=['id', 'x7'], inplace=True)

tables = [
    'cardiovascular_diseases', 'air_pollution', 'alcohol_consumption',
    'BMI', 'cholesterol', 'diabetes', 'glucose',
    'physical_activities', 'tobacco', 'country', 'time'
]

df.columns = tables
df.dropna(inplace=True)

# ===============================
# 3. Feature / Target
# ===============================
y = df['cardiovascular_diseases']
X = df.drop(columns=['cardiovascular_diseases', 'country', 'time'])

# ===============================
# 4. Train/Test split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ===============================
# 5. Lazy Training
# ===============================
reg = LazyRegressor(verbose=0, ignore_warnings=True)

models, predictions = reg.fit(
    X_train, X_test, y_train, y_test
)
models.style\
    .format({
        "R-Squared": "{:.4f}",
        "RMSE": "{:.4f}",
        "Time Taken": "{:.2f}"
    })\
    .background_gradient(cmap="Blues", subset=["R-Squared"])\
    .background_gradient(cmap="Reds", subset=["RMSE"])\
    .set_caption("LazyRegressor Model Comparison")

print(models)

  0%|          | 0/42 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000042 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 143
[LightGBM] [Info] Number of data points in the train set: 77, number of used features: 8
[LightGBM] [Info] Start training from score 45.784645
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [10]:
import plotly.express as px

models_sorted = models.sort_values(by="R-Squared", ascending=False).reset_index()
models_sorted.rename(columns={"index": "Model"}, inplace=True)

fig = px.bar(
    models_sorted.head(15),
    x="R-Squared",
    y="Model",
    orientation="h",
    title="Top 15 Models - LazyRegressor",
    hover_data=["RMSE", "Time Taken"]
)

fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()